In [1]:
import argparse
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.linalg import svd as scipy_svd
from scipy.stats import gaussian_kde
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler


/Users/yaniwu/Documents/ipsos/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## read pdf

In [4]:
cd ..

/Users/yaniwu/Documents/ipsos/Retail-Anomaly-Detection


In [5]:
df1 = pd.read_csv("data/raw/retail_processed.csv")
df1

,customer_id,city_code,active_days_normed,pack_sale_ratio_normed,daily_scan_stability_normed,scan_interval_normed,daily_brand_width_normed,inventory_sales_ratio_normed,sales_order_ratio_normed,inventory_deviation_normed,...,inventory_sales_ratio_score,sales_order_ratio_score,inventory_deviation_score,avg_transaction_qty_score,inventory_change_ratio_score,operation_match_index_score,lfm_score,pca_score,entropy_score,quality_flagged
0,cust00001230,B,0,0.076415,0.282460,-0.268875,0.157865,0.002027,-0.060792,0.038393,...,95.657859,94.988277,9.525431e+01,93.142325,75.782261,56.254992,88.939534,91.126769,88.867286,NaN
1,cust00001288,B,0,-0.459976,0.112984,-0.688882,-0.117633,0.098582,0.035021,0.038393,...,94.507164,95.293180,9.525431e+01,89.612530,88.662448,69.653182,88.056035,92.873810,89.359278,NaN
2,cust00002979,B,0,-0.586910,-0.225968,-0.240676,-0.514689,0.350322,-0.204206,0.038393,...,90.031925,92.922416,9.525431e+01,83.313402,82.183247,85.784047,87.783555,90.761997,88.931412,NaN
3,cust00000992,B,0,-0.200918,0.451936,0.013230,0.708885,0.738305,-0.096970,9.891844,...,76.986319,94.528541,2.226275e-08,87.864961,95.293650,77.550676,87.568763,85.027463,80.661502,NaN
4,cust00002388,A,0,0.057063,-0.451936,-0.729682,-0.263489,0.382737,-0.097250,-2.993844,...,89.262068,94.524831,1.028591e+00,89.624241,87.100286,73.260559,87.316987,85.060897,80.658122,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3752,cust00003435,A,-2700,-2.665437,2.429156,1.929384,-0.652438,27.483280,-3.924013,26.314262,...,0.000000,0.095975,0.000000e+00,0.906822,0.605974,45.094081,11.641704,4.087562,13.654507,NaN
3753,cust00003876,B,-1600,-2.976751,2.937584,5.426758,-1.367254,5.174168,-4.170745,0.038393,...,0.003914,0.051055,9.525431e+01,0.000009,65.904973,42.066768,11.251408,19.367827,21.965588,NaN
3754,cust00003652,B,-2100,-2.901196,6.779039,15.922457,-1.395222,4.620873,-4.170745,-6.640361,...,0.016134,0.051055,9.173964e-05,0.000025,64.057213,42.066768,10.115405,12.190168,13.425946,NaN
3755,cust00003149,A,-2700,-3.198770,2.542140,4.175746,-1.260172,40.349542,-3.997370,5.068477,...,0.000000,0.079556,5.130349e-03,0.000393,0.605974,43.301969,7.619473,2.515902,8.764288,NaN


In [3]:
pwd

'/Users/yaniwu/Documents/ipsos/Retail-Anomaly-Detection/scripts'

## LFM

## LIGHT GBM